# Ajuste de dados - tecnologia telefônica
fontes: https://informacoes.anatel.gov.br/paineis/outorga-e-licenciamento/estacoes-do-smp (para realizar isso, filtre o município ou estado de interesse e faça download da sessão de "Detalhamento de estações" --> este será o "arquivo" de input); https://informacoes.anatel.gov.br/paineis/infraestrutura/panorama (pode ter dados úteis para descrever o sistema telefônico)

In [ ]:
import pandas as pd
import os
import re
import unicodedata 

arquivo = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\_ANATEL\database.xlsx'
repopath = r'C:\Users\gabriel.coimbra\Desktop\Concórdia\_ANATEL'

cidades = ['Concórdia']


def remover_acentos(texto):
    if pd.isna(texto):
        return ''
    texto = str(texto)
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return texto

def formatar_primeiras_maiusculas(valor):
    if pd.isna(valor):
        return ''

    valor = str(valor).strip().lower()

    # Primeira letra de cada palavra em maiúsculo
    valor = valor.title()

    # Ajustes para termos que devem continuar padronizados
    substituicoes = {
        'S/N': 'S/N',
        'Sn': 'S/N',
        'Km': 'KM',
        'Br': 'BR',
        'Sc': 'SC',
        'Rs': 'RS',
        'Sp': 'SP',
        'Pr': 'PR',
        'Mg': 'MG',
        'Rua': 'Rua',
        'Av': 'Av.',
        'Avenida': 'Avenida'
    }

    for errado, certo in substituicoes.items():
        valor = re.sub(rf'\b{errado}\b', certo, valor)

    return valor
    
def limpar_endereco(valor):
    if pd.isna(valor):
        return ''

    valor = str(valor).strip()

    # Remove espaços duplicados
    valor = re.sub(r'\s+', ' ', valor)

    # Remove hífen isolado no final
    # Ex: "Rua X, 100, -" -> "Rua X, 100"
    valor = re.sub(r'\s*,?\s*-\s*$', '', valor)

    # Remove vírgula final sobrando
    valor = re.sub(r'\s*,\s*$', '', valor)

    # Padroniza espaços antes/depois de vírgula
    valor = re.sub(r'\s*,\s*', ', ', valor)

    return valor.strip()


def criar_chave_endereco(valor):
    """
    Cria chave para agrupamento.

    Ex:
    "SITIO MONTE CASTELO, S/N" 
    "Sitio Monte Castelo, S/N"
    viram a mesma chave.

    Ex:
    "Rua Brasil, S/N"
    "Rua Brasil, S/N, Lote 14, Quadra C..."
    viram a mesma chave.
    """

    valor = limpar_endereco(valor)
    valor = remover_acentos(valor).upper()

    # Padroniza S/N
    valor = re.sub(r'\bS\s*/\s*N\b', 'SN', valor)
    valor = re.sub(r'\bS\.?\s*N\.?\b', 'SN', valor)

    # Quebra por vírgula
    partes = [p.strip() for p in valor.split(',') if p.strip()]

    # Usa apenas nome da via/local + número principal
    # Ignora complementos posteriores como lote, quadra, bairro etc.
    if len(partes) >= 2:
        valor = partes[0] + ', ' + partes[1]
    elif len(partes) == 1:
        valor = partes[0]

    # Remove pontuação para comparar melhor
    valor = re.sub(r'[^\w\s]', ' ', valor)

    # Remove espaços duplicados
    valor = re.sub(r'\s+', ' ', valor).strip()

    return valor


def score_endereco(valor):
    """
    Define qual endereço deve ser mantido no resultado.
    Quanto maior o score, mais completo/preferível.
    """

    if pd.isna(valor):
        return 0

    valor_limpo = limpar_endereco(valor)
    valor_sem_acentos = remover_acentos(valor_limpo)

    score = len(valor_limpo)

    # Penaliza endereço todo em maiúsculo quando houver alternativa mais legível
    if valor_sem_acentos.isupper():
        score -= 10

    # Bonifica endereços com complemento útil
    palavras_bonus = ['LOTE', 'QUADRA', 'COLONIA', 'BAIRRO', 'KM']
    valor_upper = valor_sem_acentos.upper()

    for palavra in palavras_bonus:
        if palavra in valor_upper:
            score += 20

    return score


def escolher_endereco_mais_completo(serie):
    enderecos = [limpar_endereco(v) for v in serie if not pd.isna(v)]

    if not enderecos:
        return ''

    return formatar_primeiras_maiusculas(max(enderecos, key=score_endereco))


def juntar_unicos(serie):
    valores = []

    for v in serie:
        if pd.isna(v):
            continue

        v = str(v).strip()

        if not v or v.lower() == 'nan':
            continue

        # Caso já venha algo como "2G / 3G", separa para evitar duplicidade
        partes = [p.strip() for p in v.split('/') if p.strip()]

        for parte in partes:
            if parte not in valores:
                valores.append(parte)

    return ' / '.join(valores)


bd = pd.read_excel(arquivo)

# Limpa campos principais
bd['Endereço_limpo'] = bd['Endereço (Rua, Nº, Comp., Bairro)'].apply(limpar_endereco)
bd['Endereco_chave'] = bd['Endereço_limpo'].apply(criar_chave_endereco)

bd['Geração'] = bd['Geração'].astype(str).str.strip()
bd['Tecnologia'] = bd['Tecnologia'].astype(str).str.strip()

# Opcional: arredondar coordenadas para evitar pequenas diferenças invisíveis no Excel
bd['Latitude_aux'] = pd.to_numeric(bd['Latitude decimal'], errors='coerce').round(6)
bd['Longitude_aux'] = pd.to_numeric(bd['Longitude decimal'], errors='coerce').round(6)

bd_agrupado = (
    bd.groupby(
        [
            'Município',
            'Número Estação',
            'Operadora',
            'Latitude_aux',
            'Longitude_aux',
            'Endereco_chave'
        ],
        dropna=False
    )
    .agg({
        'Endereço (Rua, Nº, Comp., Bairro)': escolher_endereco_mais_completo,
        'Geração': juntar_unicos,
        'Tecnologia': juntar_unicos,
        'Latitude decimal': 'first',
        'Longitude decimal': 'first'
    })
    .reset_index()
)

# Remove colunas auxiliares
bd_agrupado = bd_agrupado.drop(columns=['Latitude_aux', 'Longitude_aux', 'Endereco_chave'])

# Reorganiza colunas
bd_agrupado = bd_agrupado[
    [
        'Município',
        'Número Estação',
        'Operadora',
        'Endereço (Rua, Nº, Comp., Bairro)',
        'Latitude decimal',
        'Longitude decimal',
        'Geração',
        'Tecnologia'
    ]
]

for cidade in cidades:
    bd_cidade = bd_agrupado[bd_agrupado["Município"] == cidade]

    bd_cidade.to_csv(
        os.path.join(repopath, f'tecnologia_telefonica_{cidade}.csv'),
        decimal=',',
        encoding='latin1',
        sep=';',
        index=False
    )